In [4]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [5]:
# Load the data
file_path = '2.csv'
df = pd.read_csv(file_path)

print('Shape before cleaning:', df.shape)
display(df.head())

Shape before cleaning: (1080, 11)


,genotype,yield,Relative Humidity [%],Solar Radiation [W/m2],Rainfall [mm],Wind Speed [m/s],Wind Direction [degrees],Wind Gust [m/s],Soil Temperature [C],Soil Moisture [%VWC],Soil EC [mS/cm]
0,B14A/H95,104.782508,18.449032,74.623145,262.897940,0.099130,0.447871,181.967942,2.150320,24.207225,8.041707
1,B14A/H95,118.415325,18.449032,74.623145,262.897940,0.099130,0.447871,181.967942,2.150320,24.207225,8.041707
2,B14A/H95,164.553976,18.970085,81.112559,214.338983,0.087712,0.363366,178.846482,1.562814,24.541961,10.921802
3,B14A/H95,183.848147,18.970085,81.112559,214.338983,0.087712,0.363366,178.846482,1.562814,24.541961,10.921802
4,B14A/MO17,104.199981,18.449032,74.623145,262.897940,0.099130,0.447871,181.967942,2.150320,24.207225,8.041707


In [6]:
# Inspect the dataset
print('Columns:')
for col in df.columns:
    print('-', col)

print('\nMissing values per column:')
display(df.isna().sum().to_frame('missing_count'))

Columns:
- genotype
- yield
- Relative Humidity [%]
- Solar Radiation [W/m2]
- Rainfall [mm]
- Wind Speed [m/s]
- Wind Direction [degrees]
- Wind Gust [m/s]
- Soil Temperature [C]
- Soil Moisture [%VWC]
- Soil EC [mS/cm]

Missing values per column:


,missing_count
genotype,0
yield,4
Relative Humidity [%],0
Solar Radiation [W/m2],0
Rainfall [mm],0
Wind Speed [m/s],0
Wind Direction [degrees],0
Wind Gust [m/s],0
Soil Temperature [C],0
Soil Moisture [%VWC],0


In [7]:
# Drop columns with all missing values
all_missing_cols = [col for col in df.columns if df[col].isna().all()]
print('Columns dropped because all values are missing:', all_missing_cols)

df = df.drop(columns=all_missing_cols)

# Drop rows where the target is missing
df = df.dropna(subset=['yield']).copy()

print('Shape after cleaning:', df.shape)
print('Remaining missing values:')
display(df.isna().sum().to_frame('missing_count'))

Columns dropped because all values are missing: []
Shape after cleaning: (1076, 11)
Remaining missing values:


,missing_count
genotype,0
yield,0
Relative Humidity [%],0
Solar Radiation [W/m2],0
Rainfall [mm],0
Wind Speed [m/s],0
Wind Direction [degrees],0
Wind Gust [m/s],0
Soil Temperature [C],0
Soil Moisture [%VWC],0


In [8]:
# Define features and target
X = df.drop(columns=['yield'])
y = df['yield']

categorical_features = ['genotype']
numeric_features = [col for col in X.columns if col not in categorical_features]

print('Categorical features:', categorical_features)
print('Numeric features:', numeric_features)

Categorical features: ['genotype']
Numeric features: ['Relative Humidity [%]', 'Solar Radiation [W/m2]', 'Rainfall [mm]', 'Wind Speed [m/s]', 'Wind Direction [degrees]', 'Wind Gust [m/s]', 'Soil Temperature [C]', 'Soil Moisture [%VWC]', 'Soil EC [mS/cm]']


In [9]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training set size:', X_train.shape)
print('Test set size:', X_test.shape)

Training set size: (860, 10)
Test set size: (216, 10)


In [10]:
# Preprocessing for Ridge Regression
ridge_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

ridge_model = Pipeline([
    ('preprocessor', ridge_preprocessor),
    ('model', Ridge())
])

# Preprocessing for Random Forest
rf_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

rf_model = Pipeline([
    ('preprocessor', rf_preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=1))
])

In [11]:
# Tune Ridge Regression
ridge_param_grid = {
    'model__alpha': [0.1, 1, 10, 100]
}

ridge_grid = GridSearchCV(
    estimator=ridge_model,
    param_grid=ridge_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=1
)

ridge_grid.fit(X_train, y_train)

print('Best Ridge parameters:', ridge_grid.best_params_)
print('Best cross-validation R²:', ridge_grid.best_score_)

Best Ridge parameters: {'model__alpha': 10}
Best cross-validation R²: 0.3839509350773736


In [12]:
# Tune Random Forest
rf_param_grid = {
    'model__n_estimators': [100],
    'model__max_depth': [None, 10],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=1
)

rf_grid.fit(X_train, y_train)

print('Best Random Forest parameters:', rf_grid.best_params_)
print('Best cross-validation R²:', rf_grid.best_score_)

Best Random Forest parameters: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Best cross-validation R²: 0.38988601302149445


In [13]:
# Evaluate tuned models on the test set
def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred)
    }

ridge_best = ridge_grid.best_estimator_
rf_best = rf_grid.best_estimator_

ridge_pred = ridge_best.predict(X_test)
rf_pred = rf_best.predict(X_test)

results = pd.DataFrame([
    {'Model': 'Ridge Regression', **regression_metrics(y_test, ridge_pred)},
    {'Model': 'Random Forest', **regression_metrics(y_test, rf_pred)}
])

results = results.sort_values(by='R2', ascending=False).reset_index(drop=True)
display(results)

,Model,MAE,RMSE,R2
0,Ridge Regression,29.753710,39.048747,0.345113
1,Random Forest,30.035163,39.364600,0.334476
